# 🚀 Finetuning Whisper Fongbe - Google Colab

**Workflow :**
- **Code** : GitHub (sync automatique)
- **Data** : Google Drive (persistent)
- **Models** : Google Drive (checkpoints sauvegardés)

**⚠️ Prérequis :**
1. Runtime → Change runtime type → **GPU (T4)**
2. Dataset uploadé dans Drive (voir instructions ci-dessous)

## 📦 Setup - Monter Drive & Sync Projet

In [ ]:
# 1. Monter Google Drive
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')
print("✅ Drive monté")

In [ ]:
# 2. Configuration chemins
PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
DATASET_TAR = PROJECT_ROOT / 'fongbe_dataset.tar.gz'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'fongbe_asr_unified'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
REPO_URL = 'https://github.com/Appolinairee/fongbe-asr.git'

# Créer structure si n'existe pas
for path in [DATA_ROOT, OUTPUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f"📁 Projet: {PROJECT_ROOT}")
print(f"📊 Data: {DATA_ROOT}")
print(f"💾 Outputs: {OUTPUT_ROOT}")

In [ ]:
# 3. Sync code depuis GitHub
if not (PROJECT_ROOT / '.git').exists():
    print("📥 Clone initial depuis GitHub...")
    TMP = Path('/tmp/fongbe_repo')
    subprocess.run(['git', 'clone', REPO_URL, str(TMP)], check=True)
    subprocess.run(['rsync', '-av', '--exclude=data/', '--exclude=outputs/', 
                    f'{TMP}/', str(PROJECT_ROOT)], check=True)
    subprocess.run(['rm', '-rf', str(TMP)], check=True)
    print("✅ Projet cloné")
else:
    print("🔄 Pull dernières modifications...")
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)
    print("✅ Code à jour")

In [ ]:
# 4. Extraire dataset depuis tar.gz
import tarfile

if not (DATA_ROOT / 'train').exists():
    if DATASET_TAR.exists():
        print(f"📦 Extraction dataset depuis {DATASET_TAR.name}...")
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall(PROJECT_ROOT / 'data' / 'processed')
        print("✅ Dataset extrait")
    else:
        print(f"❌ Dataset introuvable: {DATASET_TAR}")
        print("\n📤 Upload requis: fongbe_dataset.tar.gz dans Mon Drive/fongbe/")
else:
    print("✅ Dataset déjà extrait")

# Vérifier
train_path = DATA_ROOT / 'train'
if train_path.exists():
    n_files = len(list(train_path.glob('*.arrow')))
    print(f"✅ Dataset prêt: {n_files} fichiers dans train/")
else:
    print("❌ Problème extraction dataset")

## 🔧 Installation Dépendances

In [ ]:
# Installer packages nécessaires
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa
print("✅ Dépendances installées")

## 🎯 Vérification GPU

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Pas de GPU détecté! Runtime → Change runtime type → GPU")

In [ ]:
# 2. Installer dépendances
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa

In [ ]:
# 3. Monter Google Drive (si dataset là-bas)
from google.colab import drive
drive.mount('/content/drive')

# OU upload dataset directement
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# 4. Cloner repo (ou upload files)
!git clone YOUR_REPO_URL fongbe
%cd fongbe

# OU créer structure
!mkdir -p data/processed scripts

In [ ]:
# 5. Copier dataset depuis Drive (si monté)
!cp -r /content/drive/MyDrive/fongbe_data/fongbe_asr_unified data/processed/

# Vérifier
!ls -lh data/processed/fongbe_asr_unified/

In [ ]:
# 6. Code training (inline)
%%writefile scripts/train.py
import os
import torch
from datasets import load_from_disk, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MODEL_NAME = "openai/whisper-small"
DATASET_PATH = "data/processed/fongbe_asr_unified"
OUTPUT_DIR = "outputs/whisper-fongbe"

print("📦 Loading dataset...")
dataset = load_from_disk(DATASET_PATH)
dataset = dataset.cast_column("audio_path", Audio(sampling_rate=16000))
dataset = dataset.rename_column("audio_path", "audio")

print("🤖 Loading Whisper...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="yo", task="transcribe"
)

print("🔬 Applying LoRA...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["audio"]["array"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("⚙️ Preprocessing...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"],
    num_proc=2
)

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    warmup_steps=500,
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("\n🔥 Training...")
trainer.train()

print("\n📊 Final evaluation...")
metrics = trainer.evaluate(dataset["test"])
print(f"\n🎯 Test WER: {metrics['eval_wer']*100:.2f}%")

print("\n💾 Saving model...")
model.save_pretrained(f"{OUTPUT_DIR}/final")
processor.save_pretrained(f"{OUTPUT_DIR}/final")
print("✅ Done!")

In [ ]:
# 7. Lancer training
!python scripts/train.py

In [ ]:
# 8. Visualiser TensorBoard (pendant training)
%load_ext tensorboard
%tensorboard --logdir outputs/whisper-fongbe/runs

In [ ]:
# 9. Télécharger modèle final
from google.colab import files
!zip -r whisper-fongbe-final.zip outputs/whisper-fongbe/final
files.download('whisper-fongbe-final.zip')

## 🎯 Résultats attendus

- **Training time**: 3-6h sur GPU T4
- **WER baseline**: 20-40%
- **Modèle final**: ~10MB (LoRA adapters)

## 📊 Monitoring

Pendant training, regarder:
- Loss décroissante
- WER validation décroissante
- Pas d'overfitting

## 💡 Tips Colab

- Session = 12h max (gratuit)
- Sauvegarder checkpoints régulièrement
- Copier résultats vers Drive avant fin session